## Data acquisition
Notebook based on CA683-Group99 / Wind-Energy-Prediction: https://github.com/CA683-Group99/Wind-Energy-Prediction/blob/main/eirgrid_download_2026.ipynb to download the data from EirGrid Source: www.smartgriddashboard.com

In [14]:
#pip install pandas requests tenacity

In [4]:
import pandas as pd
import requests
from datetime import datetime, timedelta
import os

folder = "/Users/sbriones/Desktop/2026004_CA1/dataset/"

os.makedirs(folder, exist_ok=True)


In [5]:
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
import urllib.parse
from datetime import date

@retry(wait=wait_exponential(multiplier=2, min=4, max=64),
        stop=stop_after_attempt(10),
        retry=retry_if_exception_type(requests.exceptions.HTTPError))
def fetch_dashboard_metric_month(area: str, region: str = "ROI", start_date: date = date.today() , end_date: date = date.today()) -> pd.DataFrame:

datefrom = start_date.strftime("%d-%b-%Y 00:00")
dateto = end_date.strftime("%d-%b-%Y 23:59")


quoted_datefrom = urllib.parse.quote(datefrom)
quoted_dateto = urllib.parse.quote(dateto)

url = f"https://www.smartgriddashboard.com/DashboardService.svc/data?area={area}&region={region}&datefrom={quoted_datefrom}&dateto={quoted_dateto}"

print(f"Calling URL: {url}") # Log the URL
response = requests.get(url, timeout=15)
response.raise_for_status()
data = response.json()

if "Rows" in data and data["Rows"]:
        df = pd.DataFrame(data["Rows"])
        df['EffectiveTime'] = pd.to_datetime(df['EffectiveTime'])
        df_pivot = df.pivot_table(index='EffectiveTime', columns='FieldName', values='Value')
        print(f"DataFrame for {area} has {len(df_pivot)} rows.") # Log the number of rows
        return df_pivot

print(f"DataFrame for {area} is empty.") # Log if DataFrame is empty
    return pd.DataFrame()

@retry(wait=wait_exponential(multiplier=1, min=4, max=10),
        stop=stop_after_attempt(5),
        retry=retry_if_exception_type(requests.exceptions.HTTPError))
def _fetch_vargen_solar_chunk(region: str, start_str: str, end_str: str) -> pd.DataFrame:
    """Fetch actual solar generation from EirGrid's VarGen API"""
url = f"https://www.vargen.smartgriddashboard.com/api/export/{start_str}/{end_str}/{region}/SOLAR?FORMAT=JSON"

print(f"Calling URL: {url}") # Log the URL
response = requests.get(url, timeout=15)
response.raise_for_status()
data = response.json()

if "Rows" in data and data["Rows"]:
    df = pd.DataFrame(data["Rows"])
    if "FieldName" in df.columns:
            df = df[df["FieldName"] == "SOLAR_ACTUAL"]

        df["Timestamp"] = pd.to_datetime(df["Effective_Date"], format="%d-%m-%Y %H:%M:%S")
        df = df[["Timestamp", "MW_ACTUAL", "MW_FORECAST"]].rename(columns={"MW_ACTUAL": "solar_actual"})
        df["solar_actual"] = pd.to_numeric(df["solar_actual"], errors="coerce")

        df = df.rename(columns={"MW_FORECAST": "solar_forecast"})
        df["solar_forecast"] = pd.to_numeric(df["solar_forecast"], errors="coerce")
        print(f"DataFrame for solar_actual has {len(df)} rows.") # Log the number of rows
        return df.drop_duplicates(subset=["Timestamp"]).set_index("Timestamp")

    print(f"DataFrame for solaractual is empty.") # Log if DataFrame is empty
    return pd.DataFrame()

def fetch_vargen_solar_month(region: str = "ROI", start_date: date = date.today() , end_date: date = date.today() ) -> pd.DataFrame:
    """Fetch 15-minute actual solar generation from EirGrid's VarGen API with retry logic."""

mid_date = start_date.replace(day=16)
chunks = [
        (start_date.strftime("%Y%m%d0000"), mid_date.strftime("%Y%m%d0000")),
        (mid_date.strftime("%Y%m%d0000"),   end_date.strftime("%Y%m%d2345")),
    ]
frames = [_fetch_vargen_solar_chunk(region, s, e) for s, e in chunks]
frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames)
    return df[~df.index.duplicated(keep="first")].sort_index()

In [6]:
def build_eirgrid_month_frame(region: str = "ROI", start_date: date = date.today() , end_date: date = date.today()) -> pd.DataFrame:
"""Fetch and merge all EirGrid time-series metrics into a single DataFrame."""
metrics = ["demandactual", "generationactual", "windactual", "interconnection", "co2intensity", "co2emission"]


frames = []

    for area in metrics:
        # Using the retry-enabled function
        df = fetch_dashboard_metric_month(area, region=region, start_date=start_date, end_date=end_date)
        if not df.empty:
            frames.append(df)

# Using the retry-enabled function
solar_df = fetch_vargen_solar_month(region=region, start_date=start_date, end_date=end_date)
    if not solar_df.empty:
        frames.append(solar_df)

    if not frames:
        return pd.DataFrame()

    merged_df = frames[0]
    for df in frames[1:]:
        merged_df = merged_df.join(df, how="outer")

    column_mapping = {
        "demandactual": "SYSTEM_DEMAND",
        "windactual": "WIND_ACTUAL",
        "interconnection": "INTER_NET", 
        "co2intensity": "CO2_INTENSITY",
        "solaractual": "solar_actual", 
    }

    eirgrid_df=merged_df.rename(columns=column_mapping).sort_index()


# Define the path where you want to save the CSV file in your Google Drive
    dates = f"{start_date.strftime('%Y-%m-%d')}_{end_date.strftime('%Y-%m-%d')}"
    output_path = f"{folder}eirgrid_{region}_{dates}.csv"
    print(f"Saving DataFrame to  {output_path}") # Log the csv path

# Save the DataFrame to CSV
    eirgrid_df.to_csv(output_path, index=True) # Set index=False if you don't want to write the DataFrame index
    return eirgrid_df

In [7]:
def process_eirgrid_data(years_to_process: list, region: str) -> pd.DataFrame:
    """Fetches or loads EirGrid data for specified years and region, concatenating it into a single DataFrame."""

    def get_month_ranges(years):
        month_ranges = []
        for year in years:
            for month in range(1, 13):
                first_day = date(year, month, 1)
                if month == 12:
                    last_day = date(year + 1, 1, 1) - timedelta(days=1)
                else:
                    last_day = date(year, month + 1, 1) - timedelta(days=1)
                month_ranges.append((first_day, last_day))
        return month_ranges

    all_month_ranges = get_month_ranges(years_to_process)
    monthly_dataframes = []

    for start_date, end_date in all_month_ranges:
         dates_str = f"{start_date.strftime('%Y-%m-%d')}_{end_date.strftime('%Y-%m-%d')}"
         output_path = f"{folder}eirgrid_{region}_{dates_str}.csv"

         if (start_date > date.today()):
              print(f"Ignore future dates for {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')} from {output_path}")
         else:

          df_month = pd.DataFrame()


          is_current_month = start_date <= date.today() <= end_date

          try:
              if is_current_month:
                  raise FileNotFoundError
              print(f"Attempting to load data for {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')} from {output_path}")
              df_month = pd.read_csv(output_path, index_col=0, parse_dates=True)
              if not df_month.empty:
                  print(f"Successfully loaded data from CSV for {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}. Rows: {len(df_month)}")
              else:
                  print(f"File {output_path} loaded but was empty. Will attempt to fetch from URL.")
                  df_month = pd.DataFrame()
          except FileNotFoundError:
              print(f"File not found at {output_path}. Fetching data from URL.")
          except Exception as e:
              print(f"Error loading file {output_path}: {e}. Fetching data from URL.")
              df_month = pd.DataFrame()

          if df_month.empty:
              print(f"Fetching data from URL for: Start: {start_date.strftime('%Y-%m-%d')}, End: {end_date.strftime('%Y-%m-%d')}")
              fetched_df = build_eirgrid_month_frame(region=region, start_date=start_date , end_date=end_date)
              if not fetched_df.empty:
                  monthly_dataframes.append(fetched_df)
                  print(f"Successfully fetched and saved data for {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}. Rows: {len(fetched_df)}")
              else:
                  print(f"No data fetched from URL for {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}.")
          elif not df_month.empty:
              monthly_dataframes.append(df_month)

    if monthly_dataframes:
        full_year_df = pd.concat(monthly_dataframes).sort_index()
        print(f"\nFull year DataFrame created for {years_to_process}. Total rows: {len(full_year_df)}")
        return full_year_df
    else:
        print("No data was loaded or fetched for the specified years.")
        return pd.DataFrame()

In [8]:
test_df = fetch_dashboard_metric_month("demandactual", region="ALL",
                                       start_date=date(2014, 1, 1), end_date=date(2014, 1, 1))
test_df.head()

Calling URL: https://www.smartgriddashboard.com/DashboardService.svc/data?area=demandactual&region=ALL&datefrom=01-Jan-2014%2000%3A00&dateto=01-Jan-2014%2023%3A59
Calling URL: https://www.smartgriddashboard.com/DashboardService.svc/data?area=demandactual&region=ALL&datefrom=01-Jan-2014%2000%3A00&dateto=01-Jan-2014%2023%3A59
DataFrame for demandactual has 96 rows.


FieldName,SYSTEM_DEMAND
EffectiveTime,
2014-01-01 00:00:00,3758.0
2014-01-01 00:15:00,3724.0
2014-01-01 00:30:00,3666.0
2014-01-01 00:45:00,3611.0
2014-01-01 01:00:00,3543.0


In [12]:
START_YEAR = 2014
years_to_process = list(range(START_YEAR, date.today().year + 1))
regions = ["ALL", "ROI", "NI"]

eirgrid_data = {}
for region in regions:
    df_region = process_eirgrid_data(years_to_process, region)
    if df_region.empty:
        print(f"No data was loaded for {region}.")
        continue
    
    df_region = df_region.dropna(how="all")
    output_path = f"{folder}eirgrid_{region}_{START_YEAR}_{date.today().year}.csv"
    df_region.to_csv(output_path, index=True)
    eirgrid_data[region] = df_region

Attempting to load data for 2014-01-01 to 2014-01-31 from /Users/sbriones/Desktop/2026004_CA1/dataset/eirgrid_ALL_2014-01-01_2014-01-31.csv
Successfully loaded data from CSV for 2014-01-01 to 2014-01-31. Rows: 2976
Attempting to load data for 2014-02-01 to 2014-02-28 from /Users/sbriones/Desktop/2026004_CA1/dataset/eirgrid_ALL_2014-02-01_2014-02-28.csv
Successfully loaded data from CSV for 2014-02-01 to 2014-02-28. Rows: 2688
Attempting to load data for 2014-03-01 to 2014-03-31 from /Users/sbriones/Desktop/2026004_CA1/dataset/eirgrid_ALL_2014-03-01_2014-03-31.csv
Successfully loaded data from CSV for 2014-03-01 to 2014-03-31. Rows: 2976
Attempting to load data for 2014-04-01 to 2014-04-30 from /Users/sbriones/Desktop/2026004_CA1/dataset/eirgrid_ALL_2014-04-01_2014-04-30.csv
Successfully loaded data from CSV for 2014-04-01 to 2014-04-30. Rows: 2880
Attempting to load data for 2014-05-01 to 2014-05-31 from /Users/sbriones/Desktop/2026004_CA1/dataset/eirgrid_ALL_2014-05-01_2014-05-31.csv
